In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

clean = spark.table("urban_mobility.silver.trip_events_clean")

# 1. Rank events by lifecycle stage
stage_rank_expr = F.expr("""
    CASE event_type
        WHEN 'PAYMENT_COMPLETED' THEN 5
        WHEN 'TRIP_COMPLETED' THEN 4
        WHEN 'TRIP_CANCELLED' THEN 4
        WHEN 'TRIP_STARTED' THEN 3
        WHEN 'DRIVER_ASSIGNED' THEN 2
        WHEN 'TRIP_REQUESTED' THEN 1
        ELSE 0
    END
""")

ranked = clean.withColumn("stage_rank", stage_rank_expr)

window_spec = Window.partitionBy("trip_id").orderBy(
    F.col("stage_rank").desc(), F.col("event_timestamp").desc()
)

latest_state = (
    ranked
    .withColumn("rn", F.row_number().over(window_spec))
    .filter(F.col("rn") == 1)
    .drop("rn", "stage_rank")
)

# 2. Shape final trip-state schema
final_state = (
    latest_state
    .withColumnRenamed("event_type", "current_event_type")
    .withColumnRenamed("event_timestamp", "last_event_timestamp")
    .withColumn("updated_at", F.current_timestamp())
    .select(
        "trip_id", "driver_id", "vehicle_id", "pickup_zone_id", "dropoff_zone_id",
        "pickup_datetime", "dropoff_datetime", "distance_km",
        "fare_amount", "tip_amount", "toll_amount", "total_amount",
        "surge_multiplier", "trip_status", "payment_type",
        "current_event_type", "last_event_timestamp", "updated_at"
    )
)

print("Distinct trips (target state rows):", final_state.count())

# 3. Delta MERGE into silver.trips_current
if not spark.catalog.tableExists("urban_mobility.silver.trips_current"):
    final_state.write.format("delta").saveAsTable("urban_mobility.silver.trips_current")
    print("Bootstrap: created trips_current with", final_state.count(), "trips")
else:
    target = DeltaTable.forName(spark, "urban_mobility.silver.trips_current")
    (target.alias("t")
        .merge(final_state.alias("s"), "t.trip_id = s.trip_id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
    print("MERGE complete.")

result = spark.table("urban_mobility.silver.trips_current")
print("silver.trips_current rows:", result.count())
result.groupBy("trip_status").count().show()